#### Calculate the abnormality score using the FC data

Paper: https://doi.org/10.1038/s41591-020-0793-8

Nature Medicine paper by Ang Li et al: Calculate the funcional striatal abnormalities (FSA) scores





#### Load packages

In [1]:
import pandas as pd
import numpy as np 
import joblib
from sklearn.metrics import accuracy_score
import csv
from nilearn.input_data import NiftiLabelsMasker
import nibabel as nib
from scipy import stats, ndimage
from nilearn import image, masking
import os 
from sklearn.metrics import accuracy_score
from multiprocessing import Pool
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from nilearn.image import new_img_like
from tqdm import tqdm       # This is for a progress bar (e.g. in loops)
from sklearn.impute import SimpleImputer
from scipy.stats import ttest_ind
from itertools import combinations
from sklearn.utils.class_weight import compute_class_weight

/share/home/camilla/anaconda3/envs/env_py38/lib/python3.8/site-packages/nilearn/input_data/__init__.py:23: FutureWarning: The import path 'nilearn.input_data' is deprecated in version 0.9. Importing from 'nilearn.input_data' will be possible at least until release 0.13.0. Please import from 'nilearn.maskers' instead.
  warnings.warn(message, FutureWarning)


#### Load data

In [41]:
# Adjust dlpfc side and fALFF frequency
hemi = 'R'      # L or R
freq = '01Hz'   # 01Hz or 008Hz

# Load subject information
X_my_SZ = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[296:, :]
X_my_NC = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[0:296, :]

# Extract subject IDs
subjects_scz = X_my_SZ.iloc[:, 0]   # 319 subjects
subjects_hc = X_my_NC.iloc[:, 0]    # 296 subjects

# Load fmri data
# Run L and R DLPFC separate
ts_scz = X_my_SZ[['sub', 'filename']]   # 319 subjects
ts_hc = X_my_NC[['sub', 'filename']]    # 296 subjects

# Load FC data
fc_scz = np.load(f'/n01dat01/camilla/973/FC_BNA/SZ_FC_z_A946_{hemi}.npy')
fc_hc = np.load(f'/n01dat01/camilla/973/FC_BNA/NC_FC_z_A946_{hemi}.npy')

# Path to fALLF files
alff_scz = f'/n01dat01/camilla/973/fALFF/fALFF_scz/{freq}'             # 319 _fALFF.nii.gz files
alff_hc = f'/n01dat01/camilla/973/fALFF/fALFF_hc/{freq}'               # 296 _fALFF.nii.gz files

#### Define mask

In [2]:
dlpfc_mask_img = nib.load("/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/BN_Atlas_246_3mm.nii.gz")
dlpfc_mask_img_data = dlpfc_mask_img.get_fdata()

#### Define function for FSA calculation

Calculate the personalised FSA score

Computed as each individual's distance to the SVM hyperplane

The score is a measure of the degree of functional synchronisation between an area and the rest of the brain

In [9]:
# Define paths and new data variables
save_path = f'/n01dat01/camilla/973/FC_AbnormalityScore/{hemi}_{freq}'

subjects = X_my_SZ.iloc[:, 0].values.tolist() + X_my_NC.iloc[:, 0].values.tolist()

# Loop through subjects
for subi, sub in tqdm(enumerate(subjects)):
    # Load functional image
    if sub in ts_scz['sub'].values:
        func_path = f"{alff_scz}/{sub}_fALFF_{freq}.nii.gz"
    elif sub in ts_hc['sub'].values:
        func_path = f"{alff_hc}/{sub}_fALFF_{freq}.nii.gz"

    # Load functional image
    subject_func_img = nib.load(func_path)

    # Read the alff of BNA      (246,)
    alff_data = subject_func_img.get_fdata()
    alff_data_BNA = np.zeros(246)
    for roi in range(246):
        roi_index = np.argwhere(dlpfc_mask_img_data == roi+1) # (voxel number, 3)
        temp = 0
        for _ in range(roi_index.shape[0]):
            temp = temp + alff_data[roi_index[_,0], roi_index[_,1], roi_index[_,2]]     # (x, y, z)
        alff_data_BNA[roi] = temp / roi_index.shape[0]
    assert len(np.unique(alff_data_BNA)) > 1       # fALFF values for all 246 regions

    # FC extra-DLPFC            (246,)
    if sub in ts_scz['sub'].values:
        fc_extra = np.load(f'/n01dat01/camilla/973/FC_BNA/SZ_FC_z_A946_{hemi}.npy')[subi, :]
    elif sub in ts_hc['sub'].values:
        fc_extra = np.load(f'/n01dat01/camilla/973/FC_BNA/NC_FC_z_A946_{hemi}.npy')[subi-319, :]    # hc subjects starts at index 319
    fc_extra[np.isinf(fc_extra)] = 1

    # FC intra-DLPFC            (3,)
    if sub in ts_scz['sub'].values:
        fc_intra = np.load(f'/n01dat01/camilla/973/FC_BNA/SZ_FC_z_intraDLPFC_{hemi}.npy')[subi, :]
    elif sub in ts_hc['sub'].values:
        fc_intra = np.load(f'/n01dat01/camilla/973/FC_BNA/NC_FC_z_intraDLPFC_{hemi}.npy')[subi-319, :]
    fc_intra[np.isinf(fc_intra)] = 1

    # Concatenate features      # (495,)
    #alff_data_dlpfc = np.array([alff_data_BNA[14], alff_data_BNA[18], alff_data_BNA[20]])       # left dlpfc fALFF data
    alff_data_dlpfc = np.array([alff_data_BNA[15], alff_data_BNA[19], alff_data_BNA[21]])        # right dlpfc fALFF data
    dlpfc_features = np.concatenate([alff_data_dlpfc, fc_extra, fc_intra])
    joblib.dump(dlpfc_features, os.path.join(save_path, sub + '_candidates.pkl'))   # .pkl is a pickle file

615it [03:45,  2.73it/s]


In [11]:
# Check features
print('shape of alff:', alff_data_BNA.shape)        # (246,)
print('shape of intra_dlpfc:', fc_intra.shape)      # (3,)
print('shape of extra_dlpfc:', fc_extra.shape)      # (246,)
print('Shape of features:', dlpfc_features.shape)   # (252,)
print(dlpfc_features)

shape of alff: (246,)
shape of intra_dlpfc: (3,)
shape of extra_dlpfc: (246,)
Shape of features: (252,)
[ 5.28192210e-01  6.06987192e-01  5.35592141e-01 -7.83871140e-02
  8.71671665e-02  4.97874937e-02  1.60851321e-01 -4.18605377e-02
  5.25946097e-02 -3.18573316e-02  6.26175163e-02 -1.21681653e-01
 -9.95207597e-02  6.98481614e-02  1.74258383e-01 -7.99780682e-02
  4.24139348e-02  2.28347531e-01  2.79073741e+00 -9.73930341e-03
  1.13804458e-01  8.80429381e-02  2.30313889e-01  2.23341538e-01
  2.35957495e-01  6.08648189e-02  8.12952479e-02 -9.90438962e-03
  9.36281692e-02 -9.37773509e-03  1.24635785e-01 -6.57889195e-03
  4.29389566e-02  1.94257096e-02  1.77178630e-01  7.33169933e-02
  2.31857936e-01  1.45942357e-01  1.66385613e-01  1.21980478e-01
  2.16281776e-01  1.68825474e-01  2.35838386e-01 -1.41376199e-01
 -8.84208447e-02 -3.65743972e-02  4.53863335e-02  6.99950402e-02
  1.48086690e-02 -5.60940038e-02 -5.19491997e-02 -5.70413234e-02
 -7.46295295e-02  9.16354216e-02  2.38257611e-01 -7

#### Compute personalised FSA scores for each subject in each group

In [11]:
X_my_SZ = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[296:, :]
X_my_NC = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[0:296, :]
subjects = X_my_SZ.iloc[:, 0].values.tolist() + X_my_NC.iloc[:, 0].values.tolist()

,sub,filename,sites,age,sex,cohort,TIV
0,NC_01_0001,/n02dat01/users/ypwang/SCZ/Data/HS_suit_973/sc...,SIEMENS,25.083333,2,1,0.883920
1,NC_01_0002,/n02dat01/users/ypwang/SCZ/Data/HS_suit_973/sc...,SIEMENS,22.166667,1,1,0.545963
2,NC_01_0003,/n02dat01/users/ypwang/SCZ/Data/HS_suit_973/sc...,SIEMENS,23.000000,1,1,0.766242
3,NC_01_0004,/n02dat01/users/ypwang/SCZ/Data/HS_suit_973/sc...,SIEMENS,23.416667,2,1,0.396143
4,NC_01_0005,/n02dat01/users/ypwang/SCZ/Data/HS_suit_973/sc...,SIEMENS,24.750000,2,1,0.280202
...,...,...,...,...,...,...,...
291,NC_07_0058,/n02dat01/users/ypwang/SCZ/Data/HS_suit_973/sc...,SIEMENS,21.833333,2,7,0.532954
292,NC_07_0059,/n02dat01/users/ypwang/SCZ/Data/HS_suit_973/sc...,SIEMENS,23.000000,1,7,0.490326
293,NC_07_0060,/n02dat01/users/ypwang/SCZ/Data/HS_suit_973/sc...,SIEMENS,21.083333,1,7,0.545291
294,NC_07_0068,/n02dat01/users/ypwang/SCZ/Data/HS_suit_973/sc...,SIEMENS,30.000000,1,7,0.566478


In [4]:
cohort_list_scz = np.array(X_my_SZ['cohort']).astype(np.int16)      # [1, 2, 5, 7]
cohort_list_hc = np.array(X_my_NC['cohort']).astype(np.int16)       # [1, 2, 5, 7]
cohort_list = np.concatenate((cohort_list_scz, cohort_list_hc))

Leave-one-site-out validation for FSA score calculation

In [ ]:
# Calculate one score (biomarker indicator) for each subject.
# A biomarker for fc dlpfc difference between scz and hc
# Define paths and new data variables
hemi = 'L'      # L or R
freq = '01Hz'   # 01Hz or 008Hz
save_path = f'/n01dat01/camilla/973/FC_AbnormalityScore/{hemi}_{freq}'

# Load subject information
X_my_SZ = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[296:, :]
X_my_NC = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[0:296, :]
subjects_scz = X_my_SZ.iloc[:, 0].values.tolist()   # 319 subjects
subjects_hc = X_my_NC.iloc[:, 0].values.tolist()    # 296 subjects

# Lists to store features
dlpfc_features_scz = []
dlpfc_features_hc = []

# Loop through scz subjects
for sub in tqdm(subjects_scz):
    file_path = os.path.join(save_path, f"{sub}_candidates.pkl")
    if os.path.exists(file_path):
        dlpfc_features_scz.append(joblib.load(file_path))

# Loop through hc subjects
for sub in tqdm(subjects_hc):
    file_path = os.path.join(save_path, f"{sub}_candidates.pkl")
    if os.path.exists(file_path):
        dlpfc_features_hc.append(joblib.load(file_path))

# Convert to arrays
dlpfc_features_scz = np.array(dlpfc_features_scz)               # (319, 492)
dlpfc_features_hc = np.array(dlpfc_features_hc)                 # (296, 492)

# Standardise/normalise features for scz group
scaler_scz = StandardScaler()
dlpfc_features_std_scz = scaler_scz.fit_transform(dlpfc_features_scz)
# Replace NaN values with 0 (=no feature)
nan_indices_scz = np.isnan(dlpfc_features_std_scz)
dlpfc_features_std_scz[nan_indices_scz] = 0                             # (319, 252)

# Standardise/normalise features for hc group
scaler_hc = StandardScaler()
dlpfc_features_std_hc = scaler_hc.fit_transform(dlpfc_features_hc)
nan_indices_hc = np.isnan(dlpfc_features_std_hc)
dlpfc_features_std_hc[nan_indices_hc] = 0                               # (296, 252)

# Make loop here to: train model on 3 sites and test model on 4th sites
# Do ttest on the test sites (FSA_test_score). Repeat for all four sites
# Define sites
sites = [1, 2, 5, 7]
test_site_names = ['1', '2', '5', '7']    # for naming

# Create empty lists for t-test
t_statistics = []
p_values = []

# Generate combinations for training
train_combinations = list(combinations(sites, 3))

for i, train_sites in enumerate(train_combinations):
    test_site = [site for site in sites if site not in train_sites][0]
    print(f"Processing test site: {test_site}")
    print(f"Processing training sites: {train_sites}")

    # Filter training data
    train_indices_scz = np.where(np.isin(cohort_list_scz, train_sites))[0]
    train_indices_hc = np.where(np.isin(cohort_list_hc, train_sites))[0]
    # Filter testing data
    test_indices_scz = np.where(cohort_list_scz == test_site)[0]
    test_indices_hc = np.where(cohort_list_hc == test_site)[0]

    train_indices = np.concatenate((train_indices_scz, train_indices_hc))
    test_indices = np.concatenate((test_indices_scz, test_indices_hc))
    #print(f"Subjects in training sites: {train_indices}")
    #print(f"Subjects in testing site: {test_indices}")

    # Training data
    X_train_scz = dlpfc_features_std_scz[train_indices_scz]
    X_train_hc = dlpfc_features_std_hc[train_indices_hc]
    # Create labels for training data 
    y_train_scz = np.ones(X_train_scz.shape[0])
    y_train_hc = np.zeros(X_train_hc.shape[0])
    # Concatenate and shuffle target labels
    y_train = np.concatenate((y_train_scz, y_train_hc))

    # Compute class weights (because #subjects in scz is not equal to #subjects in hc)
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

    # Train model with class weights for training data
    X_train = np.concatenate((X_train_scz, X_train_hc), axis=0)
    svc_train = SVC(kernel='rbf', gamma='scale', C=1.0, class_weight=dict(enumerate(class_weights)))
    svc_train.fit(X_train, y_train)

    # Test data for current test site
    X_test_site_scz = dlpfc_features_std_scz[test_indices_scz]
    X_test_site_hc = dlpfc_features_std_hc[test_indices_hc]
    X_test_site = np.concatenate((X_test_site_scz, X_test_site_hc), axis=0)

    y_test_site_scz = np.ones(len(test_indices_scz))
    y_test_site_hc = np.zeros(len(test_indices_hc))
    y_test_site = np.concatenate((y_test_site_scz, y_test_site_hc))

    # Compute FSA scores for test site
    #FSA_score_test = svc_train.decision_function(X_test_site)
    FSA_score_scz = svc_train.decision_function(X_test_site_scz)
    FSA_score_hc = svc_train.decision_function(X_test_site_hc)

    # t-test
    #fsa_scores_scz = FSA_score_test[len(test_indices_scz)]
    #fsa_scores_hc = FSA_score_test[len(test_indices_hc)]
    t_stat, p_val = ttest_ind(FSA_score_scz, FSA_score_hc)
    t_statistics.append(t_stat)
    p_values.append(p_val)
    
    # Save results from t-test
    results_ttest = pd.DataFrame({
        'Cohort': [f'Cohort{test_site_names[i]}'],
        'T_statistics': [t_stat],
        'P_values': [p_val]
    })
    ttest_path = os.path.join(save_path, f'ttest_results_{test_site_names[i]}.csv')
    results_ttest.to_csv(ttest_path, index=False)

    # Save model
    model_path = os.path.join(save_path, f'SVCmodel_{test_site_names[i]}.pkl')
    joblib.dump(svc_train, model_path)

-----------------------------------------

#### Calculate the FSA score for each subject

In [81]:
# Calculate one score (biomarker indicator) for each subject.
# A biomarker for fc dlpfc difference between scz and hc
# Define paths and new data variables
hemi = 'R'      # L or R
freq = '01Hz'   # 01Hz or 008Hz

save_path = f'/n01dat01/camilla/973/FC_AbnormalityScore/{hemi}_{freq}'

# Load subject information
X_my_SZ = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[296:, :]
X_my_NC = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[0:296, :]

subjects_scz = X_my_SZ.iloc[:, 0].values.tolist()   # 319 subjects
subjects_hc = X_my_NC.iloc[:, 0].values.tolist()    # 296 subjects

# Lists to store features
dlpfc_features_scz = []
dlpfc_features_hc = []

# Loop through scz subjects
for sub in tqdm(subjects_scz):
    file_path = os.path.join(save_path, f"{sub}_candidates.pkl")
    if os.path.exists(file_path):
        dlpfc_features_scz.append(joblib.load(file_path))

# Loop through hc subjects
for sub in tqdm(subjects_hc):
    file_path = os.path.join(save_path, f"{sub}_candidates.pkl")
    if os.path.exists(file_path):
        dlpfc_features_hc.append(joblib.load(file_path))

# Convert to array
dlpfc_features_scz = np.array(dlpfc_features_scz)               # (319, 492)
dlpfc_features_hc = np.array(dlpfc_features_hc)                 # 296, 492

# Create target labels
y_scz = np.ones(len(dlpfc_features_scz))          # (319,) with only 1s
y_hc = np.zeros(len(dlpfc_features_hc))
# Combine labels
y_combined = np.concatenate((y_scz, y_hc), axis=0)

# Check classes
print('Unique classes of combined:', np.unique(y_combined))

# Standardise features for scz group
scaler_scz = StandardScaler()
dlpfc_features_std_scz = scaler_scz.fit_transform(dlpfc_features_scz)
# Replace NaN values with 0 (=no feature)
nan_indices_scz = np.isnan(dlpfc_features_std_scz)
dlpfc_features_std_scz[nan_indices_scz] = 0

# Standardise features for hc group
scaler_hc = StandardScaler()
dlpfc_features_std_hc = scaler_hc.fit_transform(dlpfc_features_hc)
# Replace NaN values with 0
nan_indices_hc = np.isnan(dlpfc_features_std_hc)
dlpfc_features_std_hc[nan_indices_hc] = 0

# Combined features
dlpfc_features_combined = np.concatenate((dlpfc_features_std_scz, dlpfc_features_std_hc), axis=0)

# Train model on both groups
svc_combined = SVC(kernel='rbf', gamma='scale', C=1.0)
svc_combined.fit(dlpfc_features_combined, y_combined)       # '.fit' trains the model
# Save model
model_path = os.path.join(save_path, 'SVCmodel_combined.pkl')
joblib.dump(svc_combined, model_path)

# Compute FSA scores
FSA_score = svc_combined.decision_function(dlpfc_features_combined)

# Save scores
subjects_combined = subjects_scz + subjects_hc
for i, sub in enumerate(subjects_combined):
    fsa_score_path = os.path.join(save_path, f"{sub}_FSA_score.txt")
    np.savetxt(fsa_score_path, [FSA_score[i]], delimiter=',')

  0%|          | 0/319 [00:00<?, ?it/s]

100%|██████████| 296/296 [00:00<00:00, 1249.29it/s]


Unique classes of combined: [0. 1.]


#### T-test

In [95]:
hemi = 'R'      # L or R
freq = '01Hz'   # 01Hz or 008Hz

# Load subject information
X_my_SZ = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[296:, :]
X_my_NC = pd.read_csv('/n02dat01/users/ypwang/SCZ/SCZ_202310/Data/Info/n00_Siemens_sites&cov_info.csv').iloc[0:296, :]

# Extract subject IDs
subjects_scz = X_my_SZ.iloc[:, 0]   # 319 subjects
subjects_hc = X_my_NC.iloc[:, 0]    # 296 subjects

# Set data path
data_path = f'/n01dat01/camilla/973/FC_AbnormalityScore/{hemi}_{freq}'

# Load data 
scz_files = [file for file in os.listdir(data_path) if 'SZ' in file and file.endswith('_FSA_score.txt')]
hc_files = [file for file in os.listdir(data_path) if 'NC' in file and file.endswith('_FSA_score.txt')]

# Create empty lists for storage
scz_scores = []
hc_scores = []

# Loop through scz subjects
for sub in subjects_scz:
    file_path = os.path.join(data_path, f"{sub}_FSA_score.txt")
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data_value = float(f.read().strip())
            scz_scores.append(data_value)

# Loop through hc subjects
for sub in subjects_hc:
    file_path = os.path.join(data_path, f"{sub}_FSA_score.txt")
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data_value = float(f.read().strip())
            hc_scores.append(data_value)

# Create dataframes
df_scz = pd.DataFrame({'FSA_score': scz_scores})
df_hc = pd.DataFrame({'FSA_score': hc_scores})

In [96]:
# Save dataframes
df_scz.to_csv(f'/n01dat01/camilla/973/FC_AbnormalityScore/{hemi}_{freq}/FSA_scores_scz.csv', index=False)
df_hc.to_csv(f'/n01dat01/camilla/973/FC_AbnormalityScore/{hemi}_{freq}/FSA_scores_hc.csv', index=False)

In [97]:
from scipy.stats import ttest_ind
# Compare FSA scores between SCZ and HC
columns = ['FSA_score']

tstats = []
pvals = []

for col in columns:
    t, p = ttest_ind(df_scz[col], df_hc[col])
    tstats.append(t)
    pvals.append(p)
    print(f"Statistical test for {col}:")
    print(f"    T-statistic: {t}")
    print(f"    P-value: {p}")
    if p < 0.05:
        print(" Result: Significant difference\n")
    else: 
        print(" Result: No significant difference\n")

# Put results into a dataframe
results_df = pd.DataFrame({
    'T_statistics': tstats,
    'P_values': pvals
})

# Save t- and p-values
results_df.to_csv(f'/n01dat01/camilla/973/FC_AbnormalityScore/{hemi}_{freq}/ttest_results.csv', index=False)
results_df

Statistical test for FSA_score:
    T-statistic: 38.73763516400501
    P-value: 6.591661336611664e-167
 Result: Significant difference



,T_statistics,P_values
0,38.737635,6.591661e-167


#### Combine results

In [98]:
L_01Hz = pd.read_csv('/n01dat01/camilla/973/FC_AbnormalityScore/L_01Hz/ttest_results.csv')
L_008Hz = pd.read_csv('/n01dat01/camilla/973/FC_AbnormalityScore/L_008Hz/ttest_results.csv')

R_01Hz = pd.read_csv('/n01dat01/camilla/973/FC_AbnormalityScore/R_01Hz/ttest_results.csv')
R_008Hz = pd.read_csv('/n01dat01/camilla/973/FC_AbnormalityScore/R_008Hz/ttest_results.csv')

data = {
    'Left_0.1Hz': L_01Hz,
    'Left_0.08Hz': L_008Hz,
    'Right_0.1Hz': R_01Hz,
    'Right_0.08Hz': R_008Hz
}

row_names = ['Left_0.1Hz', 'Left_0.08Hz', 'Right_0.1Hz', 'Right_0.08Hz']
col_names = ['Hemi_fALFFfreq', 'T_statistics', 'P_value']

# Create empty dataframe
results = pd.DataFrame(index=row_names, columns=col_names)

# Fill dataframe
for hemi, df in data.items():
    results.loc[hemi, 'Hemi_fALFFfreq'] = hemi
    results.loc[hemi, 'T_statistics'] = df['T_statistics'].values[0]
    results.loc[hemi, 'P_value'] = df['P_values'].values[0]

results.reset_index(inplace=True, drop=True)
results.to_csv('/n01dat01/camilla/973/FC_AbnormalityScore/ttest_results.csv', index=True)
results

,Hemi_fALFFfreq,T_statistics,P_value
0,Left_0.1Hz,38.618324,0.0
1,Left_0.08Hz,38.814711,0.0
2,Right_0.1Hz,38.500523,0.0
3,Right_0.08Hz,38.737635,0.0
